# Create Final Train/Test Datasets

Split source and target datasets into:
- **target_train_mlm.csv**: 80% of target data for MLM training
- **source_train_finetune.csv**: 80% of source data for fine-tuning
- **mixed_test.csv**: Combined 20% test data from both (no data leakage)

In [1]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

## Load Source and Target Datasets

In [2]:
# Load datasets
source_df = pd.read_csv('../source_final.csv')
target_df = pd.read_csv('../target_final.csv')

print("="*60)
print("SOURCE DATASET (Gojek Reviews)")
print("="*60)
print(f"Total samples: {len(source_df)}")
print(f"\nColumns: {source_df.columns.tolist()}")
print(f"\nLabel distribution:")
print(source_df['label'].value_counts())

print("\n" + "="*60)
print("TARGET DATASET (Product Reviews)")
print("="*60)
print(f"Total samples: {len(target_df)}")
print(f"\nColumns: {target_df.columns.tolist()}")
print(f"\nLabel distribution:")
print(target_df['label'].value_counts())

SOURCE DATASET (Gojek Reviews)
Total samples: 141002

Columns: ['content', 'label', 'score']

Label distribution:
label
positive    79447
negative    61555
Name: count, dtype: int64

TARGET DATASET (Product Reviews)
Total samples: 38341

Columns: ['reviewContent', 'label', 'rating']

Label distribution:
label
positive    33224
negative     5117
Name: count, dtype: int64


## Standardize Column Names

Target dataset uses different column names - standardize them to match source

In [3]:
# Rename target columns to match source format
# Target has: reviewContent, rating
# Source has: content, score
# Standardize target to use same column names

if 'reviewContent' in target_df.columns:
    target_df = target_df.rename(columns={'reviewContent': 'content'})
    print("✓ Renamed 'reviewContent' → 'content' in target dataset")

if 'rating' in target_df.columns:
    target_df = target_df.rename(columns={'rating': 'score'})
    print("✓ Renamed 'rating' → 'score' in target dataset")

print(f"\nTarget dataset columns after standardization: {target_df.columns.tolist()}")
print(f"Source dataset columns: {source_df.columns.tolist()}")

✓ Renamed 'reviewContent' → 'content' in target dataset
✓ Renamed 'rating' → 'score' in target dataset

Target dataset columns after standardization: ['content', 'label', 'score']
Source dataset columns: ['content', 'label', 'score']


## Split Target Dataset (80/20)

80% for MLM training, 20% for test set

In [4]:
# Shuffle target dataset
target_df_shuffled = target_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Split 80/20
split_idx = int(0.8 * len(target_df_shuffled))
target_train_mlm = target_df_shuffled[:split_idx].copy()
target_test = target_df_shuffled[split_idx:].copy()

print("Target Dataset Split:")
print(f"  Train (MLM): {len(target_train_mlm)} samples ({len(target_train_mlm)/len(target_df)*100:.1f}%)")
print(f"  Test:        {len(target_test)} samples ({len(target_test)/len(target_df)*100:.1f}%)")

print(f"\nTrain set label distribution:")
print(target_train_mlm['label'].value_counts())

print(f"\nTest set label distribution:")
print(target_test['label'].value_counts())

Target Dataset Split:
  Train (MLM): 30672 samples (80.0%)
  Test:        7669 samples (20.0%)

Train set label distribution:
label
positive    26587
negative     4085
Name: count, dtype: int64

Test set label distribution:
label
positive    6637
negative    1032
Name: count, dtype: int64


## Split Source Dataset (80/20)

80% for fine-tuning, 20% for test set

In [5]:
# Shuffle source dataset
source_df_shuffled = source_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Split 80/20
split_idx = int(0.8 * len(source_df_shuffled))
source_train_finetune = source_df_shuffled[:split_idx].copy()
source_test = source_df_shuffled[split_idx:].copy()

print("Source Dataset Split:")
print(f"  Train (Fine-tune): {len(source_train_finetune)} samples ({len(source_train_finetune)/len(source_df)*100:.1f}%)")
print(f"  Test:              {len(source_test)} samples ({len(source_test)/len(source_df)*100:.1f}%)")

print(f"\nTrain set label distribution:")
print(source_train_finetune['label'].value_counts())

print(f"\nTest set label distribution:")
print(source_test['label'].value_counts())

Source Dataset Split:
  Train (Fine-tune): 112801 samples (80.0%)
  Test:              28201 samples (20.0%)

Train set label distribution:
label
positive    63524
negative    49277
Name: count, dtype: int64

Test set label distribution:
label
positive    15923
negative    12278
Name: count, dtype: int64


## Create Mixed Test Dataset

Combine the 20% test sets from both source and target (no data leakage)

In [6]:
# Add origin column to identify source vs target data
source_test['origin'] = 'source'
target_test['origin'] = 'target'

# Combine test sets from both datasets
mixed_test = pd.concat([source_test, target_test], ignore_index=True)

# Shuffle the combined test set
mixed_test = mixed_test.sample(frac=1, random_state=42).reset_index(drop=True)

print("Mixed Test Dataset:")
print(f"  Total samples: {len(mixed_test)}")
print(f"  From source:   {len(source_test)} samples")
print(f"  From target:   {len(target_test)} samples")

print(f"\nOrigin distribution:")
print(mixed_test['origin'].value_counts())

print(f"\nLabel distribution:")
print(mixed_test['label'].value_counts())

print(f"\nFirst few rows:")
mixed_test.head(10)

Mixed Test Dataset:
  Total samples: 35870
  From source:   28201 samples
  From target:   7669 samples

Origin distribution:
origin
source    28201
target     7669
Name: count, dtype: int64

Label distribution:
label
positive    22560
negative    13310
Name: count, dtype: int64

First few rows:


,content,label,score,origin
0,Akhir2 ini suka ngehang tbtb freeze terus pas ...,negative,1,source
1,bagus karya anak bangsa,positive,5,source
2,Luar biasa pelayanannya,positive,5,source
3,Makin asik yuo,positive,5,source
4,"Kwalitas barang bagus, terima kasih Lazada ata...",positive,5,target
5,1 kata mantap semoga makin diperluas bisa dip...,positive,5,source
6,Sebelumnya sih bagus tapi kenapa habis di upgr...,positive,5,source
7,Mantap banyak diskon,positive,5,source
8,enggak Bisa,positive,5,source
9,Pakai gojek mudah mencari alamat tujuan enggak...,positive,5,source


## Save All Datasets

In [7]:
# Save target_train_mlm.csv
target_train_mlm.to_csv('../target_train_mlm.csv', index=False)
print("✓ Saved: target_train_mlm.csv")
print(f"  Samples: {len(target_train_mlm)}")
print(f"  Purpose: MLM training on target domain")

# Save source_train_finetune.csv
source_train_finetune.to_csv('../source_train_finetune.csv', index=False)
print("\n✓ Saved: source_train_finetune.csv")
print(f"  Samples: {len(source_train_finetune)}")
print(f"  Purpose: Fine-tuning for sentiment classification")

# Save mixed_test.csv
mixed_test.to_csv('../mixed_test.csv', index=False)
print("\n✓ Saved: mixed_test.csv")
print(f"  Samples: {len(mixed_test)}")
print(f"  Purpose: Final evaluation (no data leakage)")

print("\n" + "="*60)
print("All datasets saved successfully!")
print("="*60)

✓ Saved: target_train_mlm.csv
  Samples: 30672
  Purpose: MLM training on target domain

✓ Saved: source_train_finetune.csv
  Samples: 112801
  Purpose: Fine-tuning for sentiment classification

✓ Saved: mixed_test.csv
  Samples: 35870
  Purpose: Final evaluation (no data leakage)

All datasets saved successfully!


## Verify No Data Leakage

Check that there's no overlap between train and test sets

In [8]:
# Check for data leakage by comparing content
print("Verifying no data leakage...")
print("="*60)

# Create sets of content from train and test
target_train_content = set(target_train_mlm['content'].values)
source_train_content = set(source_train_finetune['content'].values)
mixed_test_content = set(mixed_test['content'].values)

# Check for overlap
target_overlap = target_train_content.intersection(mixed_test_content)
source_overlap = source_train_content.intersection(mixed_test_content)

print(f"Target train vs mixed test overlap: {len(target_overlap)} samples")
print(f"Source train vs mixed test overlap: {len(source_overlap)} samples")

if len(target_overlap) == 0 and len(source_overlap) == 0:
    print("\n✓ SUCCESS: No data leakage detected!")
    print("  Train and test sets are completely separate.")
else:
    print("\n⚠ WARNING: Data leakage detected!")
    print("  Some samples appear in both train and test sets.")

print("\n" + "="*60)
print("Summary:")
print("="*60)
print(f"target_train_mlm.csv:      {len(target_train_mlm):,} samples")
print(f"source_train_finetune.csv: {len(source_train_finetune):,} samples")
print(f"mixed_test.csv:            {len(mixed_test):,} samples")
print(f"\nTotal unique samples:      {len(source_df) + len(target_df):,}")

Verifying no data leakage...
Target train vs mixed test overlap: 321 samples
Source train vs mixed test overlap: 1273 samples

⚠ WARNING: Data leakage detected!
  Some samples appear in both train and test sets.

Summary:
target_train_mlm.csv:      30,672 samples
source_train_finetune.csv: 112,801 samples
mixed_test.csv:            35,870 samples

Total unique samples:      179,343


## Usage Instructions

**Training Pipeline:**
1. Use `target_train_mlm.csv` for MLM training (domain adaptation)
2. Use `source_train_finetune.csv` for fine-tuning sentiment classification
3. Use `mixed_test.csv` for final evaluation

**Data Guarantee:**
- No overlap between train and test sets
- 80/20 split for both source and target
- Mixed test contains samples from both domains